/net/pr2/projects/plgrid/plggman01/HateSpeech

In [1]:
# ============================================================
# FULL CORRECTED PYTORCH CODE
# Low-Resource Indonesian Hate Speech Detection
# ============================================================

import os
import re
import random
import warnings
import numpy as np
import pandas as pd

from collections import Counter
from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.base import BaseEstimator, ClassifierMixin

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

warnings.filterwarnings("ignore")

# ============================================================
# 1. GLOBALS
# ============================================================

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"
OUTPUT_DIR = "./paper_outputs_experiment2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")
PATH_ABUSIVE = os.path.join(DATA_DIR, "abusive.csv")

print("Using device:", DEVICE)

# ============================================================
# 2. SEED
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

# ============================================================
# 3. FILE LOADING
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception as e:
            last_error = e
    raise ValueError(f"Could not read file {path}. Last error: {last_error}")

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "not_hs", "0", "false", "no"}:
            return 0

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)

    return np.nan

def print_dataset_summary(df, name="dataset"):
    print("\n" + "=" * 70)
    print(f"DATASET: {name}")
    print("=" * 70)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    if "label" in df.columns:
        print("Label distribution:", Counter(df["label"].tolist()))

def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "IDHSD_713"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df

def load_572_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["comment_text", "tweet", "Tweet", "text", "Text"]
    possible_label_cols = ["class", "label", "Label", "HS"]

    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    label_col = next((c for c in possible_label_cols if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(f"Could not detect text/label columns in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "HS_572"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df

def load_re_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["Tweet", "tweet", "text", "Text"]
    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    if text_col is None:
        raise ValueError(f"No text column found in {path}")

    if "HS" in df.columns:
        label_col = "HS"
    elif "label" in df.columns:
        label_col = "label"
    else:
        raise ValueError(f"No HS/label column found in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "RE_13169"

    keep_cols = ["text", "label", "source"]
    if "Abusive" in df.columns:
        keep_cols.append("Abusive")

    df = df[keep_cols].dropna(subset=["text", "label"]).reset_index(drop=True)
    return df

def merge_datasets(datasets):
    merged = pd.concat(datasets, axis=0, ignore_index=True)
    merged["text"] = merged["text"].astype(str).str.strip()
    merged = merged[merged["text"].str.len() > 0].copy()
    merged = merged.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return merged

# ============================================================
# 4. TEXT NORMALIZATION
# ============================================================

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "wkwwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa",
    "xixi": "tertawa",
}

def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    if df.shape[1] < 2:
        raise ValueError("new_kamusalay.csv must have at least 2 columns.")
    col1, col2 = df.columns[:2]
    slang_dict = {}
    for _, row in df.iterrows():
        src = str(row[col1]).strip().lower()
        tgt = str(row[col2]).strip().lower()
        if src and src != "nan":
            slang_dict[src] = tgt
    return slang_dict

def load_abusive_lexicon(path):
    df = safe_read_csv(path)
    first_col = df.columns[0]
    return set(str(x).strip().lower() for x in df[first_col].dropna().tolist())

def reduce_repeated_chars(text):
    return REPEAT_CHAR_PATTERN.sub(r"\1\1", text)

def split_hashtag(match):
    return f" {match.group(1)} "

def normalize_special_tokens(text):
    tokens = text.split()
    out = []
    for tok in tokens:
        tok = LAUGHTER_VARIANTS.get(tok, tok)
        out.append(tok)
    return " ".join(out)

def basic_clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("\\n", " ")
    text = text.replace("\\/", "/")
    text = reduce_repeated_chars(text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(split_hashtag, text)
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

def slang_normalize(text, slang_dict):
    tokens = text.split()
    return " ".join([slang_dict.get(tok, tok) for tok in tokens])

def preprocess_text(text, slang_dict=None):
    text = basic_clean_text(text)
    text = normalize_special_tokens(text)
    if slang_dict is not None:
        text = slang_normalize(text, slang_dict)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

def prepare_full_dataset():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)
    abusive_words = load_abusive_lexicon(PATH_ABUSIVE)

    ds_idhsd = load_idhsd_dataset(PATH_IDHSD)
    ds_572 = load_572_dataset(PATH_572)
    ds_re = load_re_dataset(PATH_RE)

    print_dataset_summary(ds_idhsd, "IDHSD")
    print_dataset_summary(ds_572, "572 dataset")
    print_dataset_summary(ds_re, "re_dataset")

    data = merge_datasets([ds_idhsd, ds_572, ds_re])
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].copy()

    data["abusive_count"] = data["clean_text"].apply(
        lambda txt: sum(1 for tok in txt.split() if tok in abusive_words)
    )

    print_dataset_summary(
        data[["clean_text", "label"]].rename(columns={"clean_text": "text"}),
        "Merged final dataset"
    )
    return data

# ============================================================
# 5. SPLITS
# ============================================================

def make_train_val_test_split(data, test_size=0.2, val_size=0.1):
    train_df, test_df = train_test_split(
        data,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=data["label"]
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def make_low_label_split(train_df, labeled_fraction=0.05):
    labeled_df, unlabeled_df = train_test_split(
        train_df,
        train_size=labeled_fraction,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )
    return labeled_df.reset_index(drop=True), unlabeled_df.reset_index(drop=True)

# ============================================================
# 6. CLASSICAL BASELINES
# ============================================================

class TfidfFusionClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, clf_name="logreg", max_word_features=30000, max_char_features=20000):
        self.clf_name = clf_name
        self.max_word_features = max_word_features
        self.max_char_features = max_char_features

        self.word_vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=max_word_features,
            sublinear_tf=True
        )
        self.char_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=max_char_features,
            sublinear_tf=True
        )
        self.clf = None
        self.threshold_ = 0.5

    def _build_clf(self):
        if self.clf_name == "svm":
            return LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)
        if self.clf_name == "logreg":
            return LogisticRegression(max_iter=400, class_weight="balanced", random_state=RANDOM_STATE)
        if self.clf_name == "nb":
            return MultinomialNB()
        raise ValueError(f"Unknown clf_name: {self.clf_name}")

    def fit(self, X, y):
        Xw = self.word_vectorizer.fit_transform(X)
        Xc = self.char_vectorizer.fit_transform(X)
        Xf = hstack([Xw, Xc]).tocsr()
        self.clf = self._build_clf()
        self.clf.fit(Xf, y)
        return self

    def _transform(self, X):
        Xw = self.word_vectorizer.transform(X)
        Xc = self.char_vectorizer.transform(X)
        return hstack([Xw, Xc]).tocsr()

    def predict_proba_like(self, X):
        Xf = self._transform(X)
        if hasattr(self.clf, "predict_proba"):
            return self.clf.predict_proba(Xf)[:, 1]
        if hasattr(self.clf, "decision_function"):
            scores = self.clf.decision_function(Xf)
            scores = np.asarray(scores).reshape(-1)
            return 1 / (1 + np.exp(-scores))
        preds = self.clf.predict(Xf)
        return np.asarray(preds).astype(float)

    def predict(self, X):
        probs = self.predict_proba_like(X)
        return (probs >= self.threshold_).astype(int)

    def set_threshold(self, threshold):
        self.threshold_ = threshold
        return self

# ============================================================
# 7. METRICS
# ============================================================

def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="model"):
    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_prob is not None:
        try:
            result["roc_auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            result["roc_auc"] = np.nan
    else:
        result["roc_auc"] = np.nan
    return result

def tune_threshold_from_probs(y_true, y_prob):
    best_thr = 0.5
    best_score = -1.0
    for thr in np.arange(0.20, 0.81, 0.02):
        pred = (y_prob >= thr).astype(int)
        score = f1_score(y_true, pred, average="macro")
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr, best_score

# ============================================================
# 8. DATASET
# ============================================================

class TokenizedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=False,
            max_length=max_length
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def build_weighted_sampler(labels):
    class_counts = Counter(labels)
    weights = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = [weights[int(lbl)] for lbl in labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

def get_class_weights(labels):
    counts = Counter(labels)
    total = sum(counts.values())
    n_classes = len(counts)
    weights = {}
    for cls, cnt in counts.items():
        weights[int(cls)] = total / (n_classes * cnt)
    ordered = [weights.get(i, 1.0) for i in sorted(weights.keys())]
    return torch.tensor(ordered, dtype=torch.float)

# ============================================================
# 9. CUSTOM TRAINER
# ============================================================

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, train_sampler=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.train_sampler_override = train_sampler

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            class_weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        num_labels = logits.size(-1)
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self.train_sampler_override if self.train_sampler_override is not None else None,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=True,
        )

def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else np.nan,
    }

# ============================================================
# 10. TRANSFORMER TRAINING
# ============================================================

def run_transformer_model_improved(
    model_checkpoint,
    train_df,
    val_df,
    test_df,
    run_name,
    max_length=128,
    epochs=50,
    batch_size=8,
    learning_rate=2e-5,
    seed=42
):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=2
    )

    train_dataset = TokenizedTextDataset(
        train_df["clean_text"].tolist(),
        train_df["label"].tolist(),
        tokenizer=tokenizer,
        max_length=max_length
    )
    val_dataset = TokenizedTextDataset(
        val_df["clean_text"].tolist(),
        val_df["label"].tolist(),
        tokenizer=tokenizer,
        max_length=max_length
    )
    test_dataset = TokenizedTextDataset(
        test_df["clean_text"].tolist(),
        test_df["label"].tolist(),
        tokenizer=tokenizer,
        max_length=max_length
    )

    class_weights = get_class_weights(train_df["label"].tolist())
    sampler = build_weighted_sampler(train_df["label"].tolist())
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    run_output_dir = os.path.join(OUTPUT_DIR, run_name)
    os.makedirs(run_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=run_output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=seed,
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=True,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        compute_metrics=compute_metrics_for_trainer,
        class_weights=class_weights,
        train_sampler=sampler,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=50)],
    )

    trainer.train()

    val_output = trainer.predict(val_dataset)
    val_logits = val_output.predictions
    val_prob = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()
    best_thr, _ = tune_threshold_from_probs(val_df["label"], val_prob)
    val_pred = (val_prob >= best_thr).astype(int)

    test_output = trainer.predict(test_dataset)
    test_logits = test_output.predictions
    test_prob = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()
    test_pred = (test_prob >= best_thr).astype(int)

    print("\n" + "=" * 70)
    print(f"RESULTS: {run_name}")
    print("=" * 70)
    print(classification_report(test_df["label"], test_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(test_df["label"], test_pred))

    val_result = evaluate_predictions(val_df["label"], val_pred, val_prob, model_name=run_name)
    test_result = evaluate_predictions(test_df["label"], test_pred, test_prob, model_name=run_name)

    val_result.update({"split": "val", "best_threshold": best_thr, "seed": seed})
    test_result.update({"split": "test", "best_threshold": best_thr, "seed": seed})

    return pd.DataFrame([val_result, test_result]), trainer, tokenizer, best_thr

# ============================================================
# 11. SSL SELECTION
# ============================================================

def predict_transformer_probs(trainer, dataset):
    output = trainer.predict(dataset)
    logits = output.predictions
    return torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

def select_balanced_pseudo_labels(
    unlabeled_df,
    prob1,
    prob2,
    round_idx=1,
    base_threshold=0.92,
    threshold_decay=0.05,
    max_add_per_round=2000
):
    current_thr = max(0.70, base_threshold - (round_idx - 1) * threshold_decay)

    pred1 = (prob1 >= 0.5).astype(int)
    pred2 = (prob2 >= 0.5).astype(int)

    agree = pred1 == pred2
    confident = (
        ((prob1 >= current_thr) | (prob1 <= 1 - current_thr)) &
        ((prob2 >= current_thr) | (prob2 <= 1 - current_thr))
    )
    mask = agree & confident

    cand = unlabeled_df.loc[mask].copy()
    if cand.empty:
        return pd.DataFrame(), unlabeled_df.copy(), current_thr

    cand["pseudo_label"] = pred1[mask]
    cand["conf1"] = prob1[mask]
    cand["conf2"] = prob2[mask]
    cand["confidence"] = (cand["conf1"] + cand["conf2"]) / 2.0
    cand["confidence_margin"] = np.abs(cand["confidence"] - 0.5)

    per_class_cap = max_add_per_round // 2

    hs_cand = cand[cand["pseudo_label"] == 1].sort_values("confidence_margin", ascending=False).head(per_class_cap)
    non_cand = cand[cand["pseudo_label"] == 0].sort_values("confidence_margin", ascending=False).head(per_class_cap)

    selected = pd.concat([hs_cand, non_cand], ignore_index=True)

    if len(selected) < max_add_per_round:
        remaining = cand.drop(index=selected.index, errors="ignore").sort_values("confidence_margin", ascending=False)
        extra_needed = max_add_per_round - len(selected)
        selected = pd.concat([selected, remaining.head(extra_needed)], ignore_index=True)

    selected_texts = set(selected["clean_text"].tolist())
    remaining_df = unlabeled_df[~unlabeled_df["clean_text"].isin(selected_texts)].reset_index(drop=True)

    return selected.reset_index(drop=True), remaining_df, current_thr

def run_ssl_transformer_round_improved(
    labeled_df,
    unlabeled_df,
    val_df,
    test_df,
    scenario_name,
    max_rounds=2,
    max_add_per_round=2000,
    seed=42
):
    current_labeled = labeled_df.copy().reset_index(drop=True)
    current_unlabeled = unlabeled_df.copy().reset_index(drop=True)
    frames = []

    for round_idx in range(1, max_rounds + 1):
        round_name = f"{scenario_name}_ssl_round{round_idx}"
        print("\n" + "#" * 70)
        print(f"SSL ROUND: {round_name}")
        print("#" * 70)

        indobert_results, indobert_trainer, indobert_tokenizer, indobert_thr = run_transformer_model_improved(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=current_labeled,
            val_df=val_df,
            test_df=test_df,
            run_name=f"indobert_{round_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=seed
        )

        xlmr_results, xlmr_trainer, xlmr_tokenizer, xlmr_thr = run_transformer_model_improved(
            model_checkpoint="FacebookAI/xlm-roberta-base",
            train_df=current_labeled,
            val_df=val_df,
            test_df=test_df,
            run_name=f"xlmr_{round_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=seed
        )

        frames.extend([indobert_results, xlmr_results])

        pred_dataset_1 = TokenizedTextDataset(
            current_unlabeled["clean_text"].tolist(),
            [0] * len(current_unlabeled),
            tokenizer=indobert_tokenizer,
            max_length=128
        )
        pred_dataset_2 = TokenizedTextDataset(
            current_unlabeled["clean_text"].tolist(),
            [0] * len(current_unlabeled),
            tokenizer=xlmr_tokenizer,
            max_length=128
        )

        prob1 = predict_transformer_probs(indobert_trainer, pred_dataset_1)
        prob2 = predict_transformer_probs(xlmr_trainer, pred_dataset_2)

        pseudo_df, current_unlabeled, used_thr = select_balanced_pseudo_labels(
            current_unlabeled,
            prob1,
            prob2,
            round_idx=round_idx,
            base_threshold=0.92,
            threshold_decay=0.05,
            max_add_per_round=max_add_per_round
        )

        print("Used pseudo-label threshold:", used_thr)
        print("Pseudo-labeled added:", len(pseudo_df))
        print("Remaining unlabeled:", len(current_unlabeled))

        if pseudo_df.empty:
            break

        pseudo_add = pseudo_df.copy()
        if "label" in pseudo_add.columns:
            pseudo_add = pseudo_add.drop(columns=["label"])
        pseudo_add["label"] = pseudo_df["pseudo_label"].astype(int)
        pseudo_add = pseudo_add.reindex(columns=current_labeled.columns)

        current_labeled = pd.concat([current_labeled, pseudo_add], ignore_index=True)
        current_labeled = current_labeled.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ============================================================
# 12. MAIN PIPELINE
# ============================================================

def run_full_fixed_pipeline():
    data = prepare_full_dataset()
    train_df, val_df, test_df = make_train_val_test_split(data, test_size=0.2, val_size=0.1)

    scenarios = [0.05, 0.10, 0.20]
    all_results = []

    for frac in scenarios:
        scenario_name = f"{int(frac * 100)}pct"
        print("\n" + "=" * 80)
        print(f"LOW-LABEL SCENARIO: {scenario_name}")
        print("=" * 80)

        labeled_df, unlabeled_df = make_low_label_split(train_df, labeled_fraction=frac)

        indobert_results, indobert_trainer, indobert_tokenizer, indobert_thr = run_transformer_model_improved(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=labeled_df,
            val_df=val_df,
            test_df=test_df,
            run_name=f"indobert_{scenario_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5,
            seed=42
        )
        indobert_results["label_fraction"] = frac
        all_results.append(indobert_results)

        ssl_results = run_ssl_transformer_round_improved(
            labeled_df=labeled_df,
            unlabeled_df=unlabeled_df,
            val_df=val_df,
            test_df=test_df,
            scenario_name=scenario_name,
            max_rounds=2,
            max_add_per_round=max(1000, int(len(labeled_df) * 2)),
            seed=42
        )
        if not ssl_results.empty:
            ssl_results["label_fraction"] = frac
            all_results.append(ssl_results)

    final_results = pd.concat(all_results, axis=0, ignore_index=True)
    final_results.to_csv(os.path.join(OUTPUT_DIR, "final_results_fixed.csv"), index=False)

    print("\nTop TEST results by macro-F1:")
    print(
        final_results[final_results["split"] == "test"]
        .sort_values("macro_f1", ascending=False)
        [["label_fraction", "model", "accuracy", "macro_f1", "macro_precision", "macro_recall", "roc_auc", "best_threshold", "seed"]]
        .head(30)
        .to_string(index=False)
    )

    return final_results

# ============================================================
# 13. ENTRY
# ============================================================

if __name__ == "__main__":
    results = run_full_fixed_pipeline()

2026-04-18 14:26:45.854765: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda

DATASET: IDHSD
Shape: (713, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({0: 453, 1: 260})

DATASET: 572 dataset
Shape: (572, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({1: 286, 0: 286})

DATASET: re_dataset
Shape: (13169, 4)
Columns: ['text', 'label', 'source', 'Abusive']
Label distribution: Counter({0: 7608, 1: 5561})

DATASET: Merged final dataset
Shape: (14184, 2)
Columns: ['text', 'label']
Label distribution: Counter({0: 8251, 1: 5933})

LOW-LABEL SCENARIO: 5pct


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.685000,0.679710,0.488106,0.441460,0.607532,0.549817,0.644944
2,0.645700,0.648687,0.509251,0.457681,0.685703,0.574785,0.787506
3,0.546100,0.577567,0.663436,0.658460,0.734793,0.701754,0.833085
4,0.446200,0.486437,0.755066,0.755011,0.770700,0.773166,0.866957
5,0.289200,0.475985,0.756828,0.756786,0.773050,0.775271,0.882708
6,0.186000,0.469945,0.813216,0.809477,0.807894,0.811954,0.883911
7,0.064500,0.592416,0.809692,0.804498,0.804498,0.804498,0.878807
8,0.045800,0.776049,0.784141,0.783924,0.795181,0.800231,0.883710
9,0.024900,0.771073,0.808811,0.801626,0.805545,0.799019,0.883968
10,0.010200,0.801947,0.807930,0.802088,0.803097,0.801212,0.882115



RESULTS: indobert_5pct
              precision    recall  f1-score   support

           0     0.8335    0.8315    0.8325      1650
           1     0.7666    0.7692    0.7679      1187

    accuracy                         0.8054      2837
   macro avg     0.8001    0.8003    0.8002      2837
weighted avg     0.8055    0.8054    0.8055      2837

Confusion Matrix:
[[1372  278]
 [ 274  913]]

######################################################################
SSL ROUND: 5pct_ssl_round1
######################################################################


...


RESULTS: indobert_20pct_ssl_round2
              precision    recall  f1-score   support

           0     0.8949    0.8461    0.8698      1650
           1     0.8011    0.8618    0.8304      1187

    accuracy                         0.8527      2837
   macro avg     0.8480    0.8539    0.8501      2837
weighted avg     0.8556    0.8527    0.8533      2837

Confusion Matrix:
[[1396  254]
 [ 164 1023]]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.702800,0.689556,0.594714,0.588361,0.654382,0.631746,0.681474
2,0.473000,0.532751,0.786784,0.781713,0.780875,0.782735,0.843821
3,0.244900,0.569214,0.805286,0.804214,0.806576,0.814872,0.866297
4,0.191600,0.611245,0.830837,0.828409,0.826565,0.833596,0.900057
5,0.130000,0.572761,0.846696,0.842604,0.842407,0.842807,0.912378
6,0.122300,0.589672,0.793833,0.793825,0.813221,0.814466,0.922944
7,0.101100,0.616687,0.837885,0.836486,0.836253,0.845263,0.924437
8,0.090100,0.793386,0.844053,0.836496,0.847028,0.831093,0.907238
9,0.067800,0.851165,0.836123,0.834363,0.833226,0.841683,0.919464
10,0.061800,1.119639,0.821145,0.820014,0.821511,0.830279,0.914654



RESULTS: xlmr_20pct_ssl_round2
              precision    recall  f1-score   support

           0     0.9024    0.8406    0.8704      1650
           1     0.7977    0.8736    0.8339      1187

    accuracy                         0.8544      2837
   macro avg     0.8500    0.8571    0.8522      2837
weighted avg     0.8586    0.8544    0.8552      2837

Confusion Matrix:
[[1387  263]
 [ 150 1037]]


Used pseudo-label threshold: 0.87
Pseudo-labeled added: 3881
Remaining unlabeled: 885

Top TEST results by macro-F1:
 label_fraction                     model  accuracy  macro_f1  macro_precision  macro_recall  roc_auc  best_threshold  seed
           0.20     xlmr_20pct_ssl_round2  0.854424  0.852174         0.850050      0.857119 0.924587            0.40    42
           0.20 indobert_20pct_ssl_round2  0.852661  0.850070         0.847984      0.853949 0.919821            0.38    42
           0.20            indobert_20pct  0.839972  0.835436         0.835766      0.835120 0.911851            0.24    42
           0.20 indobert_20pct_ssl_round1  0.839972  0.835436         0.835766      0.835120 0.911851            0.24    42
           0.10 indobert_10pct_ssl_round1  0.829397  0.825212         0.824402      0.826147 0.898199            0.76    42
           0.10            indobert_10pct  0.829397  0.825212         0.824402      0.826147 0.898199            0.76    42
           0.20